# Image Classification with Neural Networks

Neural networks have revolutionized image classification, enabling computers to recognize and categorize images with remarkable accuracy. This notebook provides a comprehensive introduction to image classification using neural networks, from data preparation to advanced techniques like transfer learning.

## Learning Objectives

- Understand the fundamentals of image classification with neural networks
- Learn how to prepare and preprocess image data for deep learning
- Build and train a Convolutional Neural Network (CNN) for image classification
- Evaluate model performance and visualize results
- Apply advanced techniques like data augmentation and transfer learning

## 1. Import Required Libraries

Let's start by importing all the necessary libraries we'll need throughout this notebook.

In [ ]:
# Core libraries for data manipulation and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# TensorFlow and Keras for deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2

# Scikit-learn for evaluation metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Operating system interfaces
import os
import random

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

# Check for GPU availability
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

## 2. Load and Preprocess Image Data

For this tutorial, we'll use the CIFAR-10 dataset, which contains 60,000 32x32 color images in 10 classes. It's a standard benchmark dataset in machine learning and computer vision.

Let's load and preprocess the dataset:

In [ ]:
# Load CIFAR-10 dataset from Keras
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Check the shapes of the data
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Define class names for CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Normalize pixel values to be between 0 and 1
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Convert labels to one-hot encoding
y_train_onehot = to_categorical(y_train, num_classes=10)
y_test_onehot = to_categorical(y_test, num_classes=10)

# Create a validation set
validation_split = 0.2
split_idx = int(X_train.shape[0] * (1 - validation_split))

X_val = X_train[split_idx:]
y_val = y_train_onehot[split_idx:]
X_train = X_train[:split_idx]
y_train = y_train_onehot[:split_idx]

print(f"Training set: {X_train.shape[0]} images")
print(f"Validation set: {X_val.shape[0]} images")
print(f"Test set: {X_test.shape[0]} images")

## 3. Explore the Dataset

Before building our model, it's important to understand the dataset we're working with. Let's visualize some sample images from each class and examine the data distribution.

In [ ]:
# Function to show random images from each class
def plot_sample_images(X, y, class_names, n_samples_per_class=5):
    fig, axes = plt.subplots(len(class_names), n_samples_per_class, 
                             figsize=(n_samples_per_class*2, len(class_names)*2))
    
    for class_idx, class_name in enumerate(class_names):
        # Find all images of this class
        class_indices = np.where(y.flatten() == class_idx)[0]
        # Select random samples
        samples = np.random.choice(class_indices, n_samples_per_class, replace=False)
        
        for i, sample_idx in enumerate(samples):
            axes[class_idx, i].imshow(X[sample_idx])
            axes[class_idx, i].axis('off')
            
            # Only set the title for the first column
            if i == 0:
                axes[class_idx, i].set_title(class_name, fontsize=12)
    
    plt.tight_layout()
    plt.show()

# Plot sample images
y_train_labels = np.argmax(y_train, axis=1)  # Convert one-hot back to class indices
plot_sample_images(X_train, y_train_labels, class_names)

# Plot class distribution
plt.figure(figsize=(12, 5))
class_counts = np.bincount(y_train_labels)
plt.bar(class_names, class_counts)
plt.title('Class Distribution in Training Set')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

# Print some statistics about the images
print(f"Image dimensions: {X_train[0].shape}")
print(f"Pixel value range: {X_train.min()} to {X_train.max()}")
print(f"Number of classes: {len(class_names)}")

## 4. Build a Convolutional Neural Network (CNN)

Now we'll build a Convolutional Neural Network (CNN) for image classification. CNNs are particularly effective for image classification tasks because they can automatically learn spatial hierarchies of features from the input images.

Let's create a simple CNN architecture with:
- Convolutional layers with ReLU activation 
- Max pooling layers to reduce spatial dimensions
- Dropout for regularization
- Fully connected layers for classification

In [ ]:
def build_cnn_model(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    # Print model summary
    model.summary()
    
    return model

# Build the CNN model
model = build_cnn_model()

### Understanding CNN Architecture

The CNN model we've built contains several key components:

1. **Convolutional Layers**: These layers apply convolutional filters to the input image to extract features like edges, textures, and shapes.
   - First set of layers captures basic features like edges
   - Later layers learn more complex features like textures and patterns

2. **BatchNormalization**: Normalizes the activations of the previous layer, stabilizing and accelerating training.

3. **MaxPooling**: Reduces spatial dimensions while preserving important features, making the model more computationally efficient and providing some translation invariance.

4. **Dropout**: Randomly sets a fraction of inputs to zero during training, which helps prevent overfitting.

5. **Flatten**: Converts the 2D feature maps into a 1D vector for the fully connected layers.

6. **Dense Layers**: Fully connected layers that perform the classification based on the features extracted by convolutional layers.

7. **Softmax Activation**: Produces a probability distribution over the 10 classes in our dataset.

## 5. Train the Model

Now, let's compile and train our CNN model. We'll use:
- Categorical crossentropy as the loss function (standard for multi-class classification)
- Adam optimizer (efficient and widely used)
- Accuracy as our evaluation metric
- Early stopping to prevent overfitting
- Model checkpointing to save the best model

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Set up callbacks
callbacks = [
    # Stop training when validation loss doesn't improve for 10 consecutive epochs
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    # Save the best model based on validation accuracy
    keras.callbacks.ModelCheckpoint(
        filepath='best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Reduce learning rate when learning plateaus
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
epochs = 50
batch_size = 64

history = model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

# Load the best model
model = keras.models.load_model('best_model.h5')

## 6. Evaluate Model Performance

Now that we've trained our model, let's evaluate its performance on the test set and analyze various metrics.

In [ ]:
# Evaluate model on test data
test_loss, test_acc = model.evaluate(X_test, y_test_onehot, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Make predictions on test set
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test_onehot, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Calculate per-class accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
plt.figure(figsize=(12, 6))
plt.bar(class_names, per_class_accuracy * 100)
plt.xlabel('Class')
plt.ylabel('Accuracy (%)')
plt.title('Per-Class Accuracy')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.show()

# Find most misclassified classes
misclassification = {}
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j:
            misclassification[(class_names[i], class_names[j])] = cm[i, j]

# Sort by number of misclassifications
top_misclassifications = sorted(misclassification.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 misclassifications:")
for (true_class, pred_class), count in top_misclassifications:
    print(f"True: {true_class}, Predicted: {pred_class}, Count: {count}")

## 7. Visualize Results

Let's visualize the training process and examine our model's learning patterns.

In [ ]:
# Plot training & validation accuracy and loss values
plt.figure(figsize=(16, 6))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Visualize feature maps from convolutional layers
def display_activation_maps(model, img_idx, layer_names=None):
    """Display activation maps for specified layers"""
    if layer_names is None:
        # Get names of convolutional layers
        layer_names = [layer.name for layer in model.layers if 'conv2d' in layer.name]
    
    # Create a model that outputs all layer activations
    activation_model = models.Model(
        inputs=model.input,
        outputs=[model.get_layer(layer_name).output for layer_name in layer_names]
    )
    
    # Choose an image and get its activations
    img = X_test[img_idx:img_idx+1]
    activations = activation_model.predict(img)
    
    # Display original image
    plt.figure(figsize=(4, 4))
    plt.imshow(X_test[img_idx])
    plt.title(f"Original Image (Class: {class_names[y_true[img_idx]]})")
    plt.axis('off')
    plt.show()
    
    # Display activation maps for each layer
    for i, (layer_name, layer_activation) in enumerate(zip(layer_names, activations)):
        # Only plot a subset of feature maps for readability
        n_features = layer_activation.shape[-1]
        n_cols = 8
        n_rows = min(8, (n_features + n_cols - 1) // n_cols)
        n_display = n_rows * n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
        fig.suptitle(f"Activation Maps for Layer: {layer_name}")
        
        for idx in range(n_display):
            row, col = idx // n_cols, idx % n_cols
            if n_rows == 1:
                ax = axes[col]
            elif n_cols == 1:
                ax = axes[row]
            else:
                ax = axes[row, col]
            
            if idx < n_features:
                # Display feature map
                ax.imshow(layer_activation[0, :, :, idx], cmap='viridis')
            ax.set_xticks([])
            ax.set_yticks([])
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        plt.show()

# Choose a random image and visualize its feature maps
random_img_idx = np.random.randint(0, len(X_test))
# Select just the first conv layer and one from the middle
layer_names = [model.layers[0].name, model.layers[6].name]
display_activation_maps(model, random_img_idx, layer_names)

## 8. Make Predictions on New Images

Let's demonstrate how to use our trained model to make predictions on individual images from the test set.

In [ ]:
# Function to display image with prediction
def show_prediction(model, X, y_true, class_names, idx):
    # Get the image and make prediction
    img = X[idx:idx+1]
    pred = model.predict(img)[0]
    pred_class = np.argmax(pred)
    true_class = y_true[idx]
    
    # Display image with prediction information
    plt.figure(figsize=(6, 6))
    plt.imshow(X[idx])
    plt.axis('off')
    
    # Determine if prediction is correct
    prediction_result = "CORRECT" if pred_class == true_class else "WRONG"
    color = "green" if pred_class == true_class else "red"
    
    plt.title(f"True: {class_names[true_class]}\nPredicted: {class_names[pred_class]} ({prediction_result})",
              color=color, fontsize=14)
    
    # Show prediction probabilities for all classes
    plt.figure(figsize=(10, 5))
    plt.bar(class_names, pred, color=[color if i == pred_class else 'skyblue' for i in range(len(class_names))])
    plt.xlabel('Class')
    plt.ylabel('Probability')
    plt.title('Prediction Probabilities')
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

# Display predictions for a few random images
for _ in range(5):
    random_idx = np.random.randint(0, len(X_test))
    show_prediction(model, X_test, y_true, class_names, random_idx)

## 9. Image Augmentation Techniques

Data augmentation is a technique to artificially expand the training dataset by creating modified versions of images. This helps improve model generalization and reduces overfitting.

Let's implement data augmentation and train a new model to see if it improves performance.

In [ ]:
# Create an image data generator with augmentation
datagen = ImageDataGenerator(
    rotation_range=15,          # randomly rotate images by up to 15 degrees
    width_shift_range=0.1,      # randomly shift images horizontally by up to 10%
    height_shift_range=0.1,     # randomly shift images vertically by up to 10%
    horizontal_flip=True,       # randomly flip images horizontally
    zoom_range=0.1,             # randomly zoom images by up to 10%
    fill_mode='nearest'         # strategy for filling newly created pixels
)

# Visualize what augmented images look like
def plot_augmented_images(X, y, datagen, class_idx, n_examples=5):
    # Get images of a specific class
    class_images = X[y.flatten() == class_idx]
    example_image = class_images[0:1]
    
    plt.figure(figsize=(12, 3))
    plt.subplot(1, n_examples+1, 1)
    plt.imshow(example_image[0])
    plt.title("Original")
    plt.axis('off')
    
    # Generate augmented images
    aug_iter = datagen.flow(example_image, batch_size=1)
    for i in range(n_examples):
        augmented_image = aug_iter.next()[0]
        plt.subplot(1, n_examples+1, i+2)
        plt.imshow(augmented_image)
        plt.title(f"Augmented #{i+1}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Show augmentation examples for a few classes
for class_idx in [0, 2, 5]:  # airplane, bird, dog
    plot_augmented_images(X_train, y_train_labels, datagen, class_idx)

# Build a new model with the same architecture as before
augmented_model = build_cnn_model()
augmented_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Set up callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(filepath='best_augmented_model.h5', 
                                  monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
]

# Fit the model with data augmentation
# Note: This part can be time-consuming. If you want to skip it, comment it out.
"""
epochs = 50
batch_size = 64

# Initialize data generator
datagen.fit(X_train)

# Train the model with augmented data
augmented_history = augmented_model.fit(
    datagen.flow(X_train, y_train, batch_size=batch_size),
    epochs=epochs,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    steps_per_epoch=len(X_train) // batch_size,
    verbose=1
)

# Load the best model
augmented_model = keras.models.load_model('best_augmented_model.h5')

# Compare performance of both models
print("Performance without augmentation:")
test_loss, test_acc = model.evaluate(X_test, y_test_onehot, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

print("\nPerformance with augmentation:")
aug_test_loss, aug_test_acc = augmented_model.evaluate(X_test, y_test_onehot, verbose=0)
print(f"Test accuracy: {aug_test_acc:.4f}")
"""

## 10. Transfer Learning Implementation

Transfer learning involves using a pre-trained model (usually trained on a large dataset like ImageNet) as a starting point for a new task. This is particularly useful when you have limited training data.

Let's implement transfer learning with a few popular architectures:
- VGG16
- ResNet50
- MobileNetV2

In [ ]:
# Note: CIFAR-10 images are 32x32 pixels, but most pre-trained models expect larger images
# Let's resize our images to 224x224 (standard input size for many pre-trained models)

def resize_dataset(X, target_size=(224, 224)):
    resized_X = np.zeros((X.shape[0], *target_size, 3))
    for i in range(X.shape[0]):
        # Use TensorFlow's resize function
        resized_X[i] = tf.image.resize(X[i], target_size)
    return resized_X

# Resize a small subset for demonstration purposes
# In practice, you would resize the entire dataset
n_samples = 1000
X_train_resized = resize_dataset(X_train[:n_samples])
X_val_resized = resize_dataset(X_val[:500])
X_test_resized = resize_dataset(X_test[:500])

y_train_transfer = y_train[:n_samples]
y_val_transfer = y_val[:500]
y_test_transfer = y_test_onehot[:500]

print(f"Resized image shape: {X_train_resized[0].shape}")

# Function to create a transfer learning model
def build_transfer_model(base_model_name, input_shape=(224, 224, 3), num_classes=10):
    # Select the base model
    if base_model_name == 'vgg16':
        base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'resnet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'mobilenetv2':
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        raise ValueError(f"Unsupported model: {base_model_name}")
    
    # Freeze the base model layers
    base_model.trainable = False
    
    # Create a new model on top
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Create a transfer learning model using VGG16
# Note: This part can be time-consuming. If you want to skip it, comment it out.
"""
# Create transfer learning model
transfer_model = build_transfer_model('vgg16')
transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
transfer_model.summary()

# Train the model (just a few epochs for demonstration)
transfer_history = transfer_model.fit(
    X_train_resized, y_train_transfer,
    batch_size=32,
    epochs=5,
    validation_data=(X_val_resized, y_val_transfer),
    verbose=1
)

# Evaluate on test set
transfer_test_loss, transfer_test_acc = transfer_model.evaluate(
    X_test_resized, y_test_transfer, verbose=1
)
print(f"Transfer learning test accuracy: {transfer_test_acc:.4f}")
"""

# Compare performance across different models
models_comparison = pd.DataFrame({
    'Model': ['CNN', 'CNN with Augmentation', 'Transfer Learning (VGG16)'],
    'Test Accuracy': [0.75, 0.78, 0.83],  # Example values
    'Training Time': ['Medium', 'Long', 'Short'],
    'Model Size': ['Small (~10MB)', 'Small (~10MB)', 'Large (~500MB)']
})

print("\nComparison of Different Models:")
print(models_comparison)

## Conclusion

In this notebook, we've explored image classification with neural networks using the CIFAR-10 dataset. We've covered:

1. **Data preparation** - Loading, exploring, and preprocessing image data
2. **CNN architecture** - Building a convolutional neural network for image classification
3. **Model training** - Training the model with appropriate loss functions and optimizers
4. **Performance evaluation** - Using various metrics to evaluate model performance
5. **Visualization** - Visualizing results, feature maps, and model predictions
6. **Advanced techniques** - Data augmentation and transfer learning

Image classification is a fundamental task in computer vision, and neural networks (particularly CNNs) have dramatically improved the state-of-the-art in this field. As you continue learning, you might want to explore more advanced architectures, larger datasets, and specialized techniques for specific types of images.

### Next Steps

- Try the model on your own custom images
- Experiment with different CNN architectures (deeper or wider networks)
- Implement techniques like Grad-CAM to visualize what regions of the image the model focuses on
- Fine-tune transfer learning models by unfreezing some layers
- Explore object detection and image segmentation as extensions of classification